# **mBERT FINE-TUNED ENGLISH CLASSIFICATION**

In [ ]:
# ============================================================
# BLOCK 0 — INSTALL
# ============================================================
!pip install -q transformers accelerate scikit-learn safetensors

# ============================================================
# BLOCK 1 — IMPORTS
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from collections import OrderedDict
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from torch.utils.data import Dataset, DataLoader

print("Imports done.")


# ============================================================
# BLOCK 2 — REPRODUCIBILITY
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds set.")


# ============================================================
# BLOCK 3 — LOAD + CLEAN DATA
# ============================================================
DATA_PATH = "final_combined.csv"
for candidate in [DATA_PATH, "/content/final_combined.csv", "/mnt/data/final_combined.csv"]:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

df = None
for encoding in ["utf-8", "latin-1", "cp1252", "utf-8-sig"]:
    try:
        df = pd.read_csv(DATA_PATH, encoding=encoding)
        print(f"Loaded with encoding: {encoding}")
        break
    except UnicodeDecodeError:
        print(f"Failed with encoding: {encoding}")
        continue

if df is None:
    raise ValueError("Could not load CSV with any encoding.")

required_cols = ["sentence", "label", "distortion", "language"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

df = df.dropna(subset=required_cols).copy()
df["sentence"]   = df["sentence"].astype(str).str.strip()
df["label"]      = df["label"].astype(int)
df["distortion"] = df["distortion"].astype(str).str.strip()
df["language"]   = df["language"].astype(str).str.strip()

before = len(df)
df = df.drop_duplicates(subset=["sentence"]).copy()
df = df[df["sentence"].str.len() > 0].copy()
df = df.reset_index(drop=True)

print(f"Removed {before - len(df)} duplicates. Dataset size: {len(df)}")
print("Label distribution:\n", df["label"].value_counts())
print("Language distribution:\n", df["language"].value_counts())
print("Top distortions:\n", df["distortion"].value_counts().head(10))

df["length"] = df["sentence"].str.split().str.len()
print("\nLength stats by label:")
print(df.groupby("label")["length"].describe()[["mean","50%","min","max"]])
df = df.drop(columns=["length"])

print("Data loaded.")


In [ ]:
# ============================================================
# BLOCK 4 — TOKENIZER + COLLATOR
# ============================================================
MODEL_NAME    = "bert-base-multilingual-cased"
tokenizer     = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Tokenizer loaded: {MODEL_NAME}")


# ============================================================
# BLOCK 5 — DATASET CLASS
# ============================================================
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_col, label_col, max_len=128):
        self.texts     = dataframe[text_col].tolist()
        self.labels    = dataframe[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

print("TextDataset defined.")


# ============================================================
# BLOCK 6 — METRICS
# ============================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

print("compute_metrics defined.")


# ============================================================
# BLOCK 7 — WEIGHTED TRAINER
# ============================================================
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = (
            torch.tensor(class_weights, dtype=torch.float)
            if class_weights is not None else None
        )

    def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
        import safetensors.torch as st
        ckpt_file = os.path.join(resume_from_checkpoint, "model.safetensors")
        if os.path.exists(ckpt_file):
            try:
                tensors = st.load_file(ckpt_file)
                fixed, renamed = OrderedDict(), 0
                for k, v in tensors.items():
                    new_k = k.replace(".gamma", ".weight").replace(".beta", ".bias")
                    if new_k != k:
                        renamed += 1
                    fixed[new_k] = v
                if renamed > 0:
                    st.save_file(fixed, ckpt_file)
                    print(f"  [checkpoint fix] Renamed {renamed} keys")
            except Exception as e:
                print(f"  [checkpoint fix] Skipped: {e}")
        super()._load_from_checkpoint(resume_from_checkpoint, model=model)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits  = outputs.get("logits")
        if self.class_weights is None:
            loss_fct = torch.nn.CrossEntropyLoss()
        else:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer defined.")


# ============================================================
# BLOCK 8 — CHECKPOINT KEY FIXER
# ============================================================
def fix_checkpoint_keys(model_dir):
    try:
        import safetensors.torch as st
        path = os.path.join(model_dir, "model.safetensors")
        if not os.path.exists(path):
            return
        tensors = st.load_file(path)
        fixed, renamed = OrderedDict(), 0
        for k, v in tensors.items():
            new_k = k.replace(".gamma", ".weight").replace(".beta", ".bias")
            if new_k != k:
                renamed += 1
            fixed[new_k] = v
        st.save_file(fixed, path)
        print(f"  [fix] Fixed {renamed} keys in {path}")
    except Exception as e:
        print(f"  [fix] Skipped: {e}")

print("fix_checkpoint_keys defined.")


# ============================================================
# BLOCK 9 — SANITY CHECKS
# ============================================================
print("\n========== SANITY CHECKS ==========")
print("Null sentences    :", df["sentence"].isnull().sum())
print("Remaining dupes   :", df["sentence"].duplicated().sum())
bin_counts = df["label"].value_counts()
print(f"Class ratio       : {bin_counts.max()/bin_counts.min():.2f}:1")
dist_counts = df[df["label"]==1]["distortion"].value_counts()
print("Distortion counts :\n", dist_counts)
print("====================================\n")


In [ ]:
# ============================================================
# BLOCK 10 — BINARY DATA PREPARATION
# ============================================================
bin_df = df.copy()
bin_df["stratify_key"] = (
    bin_df["label"].astype(str) + "_" + bin_df["language"].astype(str)
)

valid_keys = bin_df["stratify_key"].value_counts()
bin_df = bin_df[bin_df["stratify_key"].isin(valid_keys[valid_keys > 1].index)]

train_bin_df, val_bin_df = train_test_split(
    bin_df,
    test_size=0.2,
    random_state=SEED,
    stratify=bin_df["stratify_key"],
)

overlap = set(train_bin_df["sentence"]) & set(val_bin_df["sentence"])
print(f"[BINARY] Sentence overlap : {len(overlap)}  (must be 0)")
print(f"[BINARY] Train: {len(train_bin_df)} | Val: {len(val_bin_df)}")
print("[BINARY] Train label dist BEFORE balance:\n",
      train_bin_df["label"].value_counts())

df_majority = train_bin_df[train_bin_df["label"] == 0].copy()
df_minority  = train_bin_df[train_bin_df["label"] == 1].copy()

print(f"\n[BINARY] Before resampling:")
print(f"  Majority : {len(df_majority)}")
print(f"  Minority : {len(df_minority)}")
print(f"  Ratio    : {len(df_majority)/len(df_minority):.2f}:1")

target_majority = len(df_minority) * 2
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=min(target_majority, len(df_majority)),
    random_state=SEED,
)

train_bin_df_balanced = pd.concat([
    df_majority_downsampled,
    df_minority,
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\n[BINARY] After resampling:")
print(f"  Majority : {len(df_majority_downsampled)}")
print(f"  Minority : {len(df_minority)}  (untouched)")
print(f"  Ratio    : {len(df_majority_downsampled)/len(df_minority):.2f}:1")
print(f"  Total    : {len(train_bin_df_balanced)}")
print("\n[BINARY] Train label dist AFTER balance:\n",
      train_bin_df_balanced["label"].value_counts())
print("[BINARY] Val label dist (untouched):\n",
      val_bin_df["label"].value_counts())

train_bin_dataset = TextDataset(
    train_bin_df_balanced, tokenizer, "sentence", "label", max_len=128
)
val_bin_dataset = TextDataset(
    val_bin_df, tokenizer, "sentence", "label", max_len=128
)

print("Binary datasets ready.")

In [ ]:
# ============================================================
# BLOCK 11 — BINARY MODEL + TRAINING ARGS
# ============================================================
num_bin_labels = 2
bin_weights    = np.array([1.0, 2.0])
print("[BINARY] Class weights:", bin_weights)

bin_config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_bin_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)
bin_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=bin_config
)

bin_total_steps  = (len(train_bin_dataset) // (16 * 2)) * 15
bin_warmup_steps = int(bin_total_steps * 0.20)
print(f"[BINARY] Total steps: {bin_total_steps} | Warmup steps: {bin_warmup_steps}")

bin_args = TrainingArguments(
    output_dir="./results_binary",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    warmup_steps=bin_warmup_steps,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.05,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
    seed=SEED,
    report_to="none",
)

bin_trainer = WeightedTrainer(
    model=bin_model,
    args=bin_args,
    train_dataset=train_bin_dataset,
    eval_dataset=val_bin_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    class_weights=bin_weights,
)

print("Binary model + trainer ready.")

# ============================================================
# BLOCK 12 — BINARY TRAINING
# ============================================================
print("\n====================")
print("Training BINARY model")
print("====================")
bin_trainer.train()

print("\n[BINARY] Fixing checkpoint keys...")
for ckpt in os.listdir("./results_binary"):
    ckpt_path = os.path.join("./results_binary", ckpt)
    if os.path.isdir(ckpt_path):
        fix_checkpoint_keys(ckpt_path)

bin_trainer.save_model("./best_binary_model")
tokenizer.save_pretrained("./best_binary_model")
fix_checkpoint_keys("./best_binary_model")
print("Binary model saved to ./mbert_best_binary_model")

In [ ]:
# ============================================================
# BLOCK 13 — BINARY EVALUATION
# ============================================================
bin_model_eval = AutoModelForSequenceClassification.from_pretrained(
    "./best_binary_model"
)
bin_model_eval.eval()

bin_pred = Trainer(
    model=bin_model_eval,
    args=TrainingArguments(
        output_dir="./eval_tmp_bin",
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    data_collator=data_collator,
).predict(val_bin_dataset)

bin_y_true = bin_pred.label_ids
bin_logits = bin_pred.predictions
bin_y_pred = bin_logits.argmax(-1)

precision, recall, f1, _ = precision_recall_fscore_support(
    bin_y_true, bin_y_pred, average="weighted", zero_division=0
)
acc = accuracy_score(bin_y_true, bin_y_pred)

print(f"\n[BINARY] accuracy={acc:.4f} | precision={precision:.4f} | "
      f"recall={recall:.4f} | f1={f1:.4f}")
print("\n[BINARY] Prediction distribution:")
print(dict(zip(*[x.tolist() for x in np.unique(bin_y_pred, return_counts=True)])))
print("\n[BINARY] Confusion matrix:")
print(confusion_matrix(bin_y_true, bin_y_pred))
print("\n[BINARY] Classification report (threshold=0.50):")
print(classification_report(bin_y_true, bin_y_pred,
      target_names=["No Distortion", "Distorted"], digits=4))


# ============================================================
# BLOCK 14 — BINARY THRESHOLD TUNING
# ============================================================
print("\n========== THRESHOLD TUNING ==========")

bin_probs    = torch.softmax(
    torch.tensor(bin_logits, dtype=torch.float), dim=-1
).numpy()
class1_probs = bin_probs[:, 1]
thresholds   = np.arange(0.1, 0.9, 0.05)
results      = []

for t in thresholds:
    preds_t = (class1_probs >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(
        bin_y_true, preds_t,
        average=None, zero_division=0, labels=[0, 1]
    )
    results.append({
        "threshold":        round(float(t), 2),
        "class1_precision": round(float(p[1]), 4),
        "class1_recall":    round(float(r[1]), 4),
        "class1_f1":        round(float(f[1]), 4),
        "class0_recall":    round(float(r[0]), 4),
    })

results_df     = pd.DataFrame(results)
print(results_df.to_string(index=False))

best_row       = results_df.loc[results_df["class1_f1"].idxmax()]
best_threshold = float(best_row["threshold"])

print(f"\n Best threshold : {best_threshold}")
print(f"   Class 1 precision : {best_row['class1_precision']}")
print(f"   Class 1 recall    : {best_row['class1_recall']}")
print(f"   Class 1 F1        : {best_row['class1_f1']}")
print(f"   Class 0 recall    : {best_row['class0_recall']}")

with open("./best_binary_model/threshold.json", "w") as f:
    json.dump({"threshold": best_threshold}, f, indent=2)
print(f"[BINARY] Threshold {best_threshold} saved.")

best_preds = (class1_probs >= best_threshold).astype(int)
print(f"\n[BINARY] Classification report at threshold={best_threshold}:")
print(classification_report(bin_y_true, best_preds,
      target_names=["No Distortion", "Distorted"], digits=4))
print("======================================\n")

In [ ]:
# ============================================================
# BLOCK 15 — MULTICLASS DATA PREPARATION
# ============================================================
dist_df = df[df["label"] == 1].copy()
dist_df = dist_df[
    ~dist_df["distortion"].str.lower().isin(["no distortion", "none", ""])
].copy()

print("[MULTICLASS] All class counts:")
print(dist_df["distortion"].value_counts())

distortion_classes = sorted(dist_df["distortion"].unique().tolist())
dist2id = {d: i for i, d in enumerate(distortion_classes)}
id2dist = {i: d for d, i in dist2id.items()}
dist_df["dist_label"] = dist_df["distortion"].map(dist2id)

assert dist_df["dist_label"].isnull().sum() == 0, "Null labels found!"
print(f"\n[MULTICLASS] Num classes : {len(distortion_classes)}")
print(f"[MULTICLASS] Classes     : {distortion_classes}")

dist_df["stratify_key"] = (
    dist_df["dist_label"].astype(str) + "_" + dist_df["language"].astype(str)
)
valid_keys = dist_df["stratify_key"].value_counts()
dist_df = dist_df[dist_df["stratify_key"].isin(valid_keys[valid_keys > 1].index)]

train_dist_df, val_dist_df = train_test_split(
    dist_df,
    test_size=0.2,
    random_state=SEED,
    stratify=dist_df["stratify_key"],
)

print(f"\n[MULTICLASS] Train: {len(train_dist_df)} | Val: {len(val_dist_df)}")
print("[MULTICLASS] Train dist BEFORE balance:\n",
      train_dist_df["distortion"].value_counts())

class_counts_train = train_dist_df["distortion"].value_counts()
median_size        = int(class_counts_train.median())

print(f"\n[MULTICLASS] Median class size : {median_size}")
print(f"[MULTICLASS] Resampling plan:")

resampled_dfs = []
for distortion_name in distortion_classes:
    class_df   = train_dist_df[
        train_dist_df["distortion"] == distortion_name
    ].copy()
    class_size = len(class_df)

    if class_size >= median_size:
        resampled = class_df
        action    = f"kept {class_size}  (no change)"

    elif class_size >= 200:
        target    = median_size
        resampled = resample(
            class_df, replace=True,
            n_samples=target, random_state=SEED
        )
        action = f"oversampled {class_size} → {target}"

    else:
        target    = min(int(class_size * 2.5), median_size)
        resampled = resample(
            class_df, replace=True,
            n_samples=target, random_state=SEED
        )
        action = f"oversampled {class_size} → {target}  (2.5x)"

    print(f"  {distortion_name:<30} : {action}")
    resampled_dfs.append(resampled)

train_dist_df_balanced = pd.concat(resampled_dfs).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

print(f"\n[MULTICLASS] Train dist AFTER balance:\n",
      train_dist_df_balanced["distortion"].value_counts())
print(f"[MULTICLASS] Total train samples : {len(train_dist_df_balanced)}")
print("[MULTICLASS] Val dist (untouched):\n",
      val_dist_df["distortion"].value_counts())

train_dist_dataset = TextDataset(
    train_dist_df_balanced, tokenizer, "sentence", "dist_label", max_len=128
)
val_dist_dataset = TextDataset(
    val_dist_df, tokenizer, "sentence", "dist_label", max_len=128
)

print("Multiclass datasets ready.")

In [ ]:
# ============================================================
# BLOCK 16 — MULTICLASS MODEL + TRAINING ARGS
# ============================================================
num_dist_labels = len(distortion_classes)
print(f"[MULTICLASS] Num classes : {num_dist_labels}")

dist_config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_dist_labels,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)
dist_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=dist_config
)

dist_total_steps  = (len(train_dist_dataset) // (8 * 4)) * 30
dist_warmup_steps = int(dist_total_steps * 0.10)

dist_args = TrainingArguments(
    output_dir="./results_distortion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,              
    warmup_steps=dist_warmup_steps,  
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=30,
    weight_decay=0.05,
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,  
    seed=SEED,
    report_to="none",
)

dist_trainer = Trainer(
    model=dist_model,
    args=dist_args,
    train_dataset=train_dist_dataset,
    eval_dataset=val_dist_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=7)],
)

print("Multiclass model + trainer ready.")


# ============================================================
# BLOCK 17 — MULTICLASS TRAINING
# ============================================================
print("\n====================")
print("Training MULTICLASS model")
print("====================")
dist_trainer.train()

print("\n[MULTICLASS] Fixing checkpoint keys...")
for ckpt in os.listdir("./results_distortion"):
    ckpt_path = os.path.join("./results_distortion", ckpt)
    if os.path.isdir(ckpt_path):
        fix_checkpoint_keys(ckpt_path)

dist_trainer.save_model("./best_distortion_model")
tokenizer.save_pretrained("./best_distortion_model")
fix_checkpoint_keys("./best_distortion_model")

with open("./best_distortion_model/label_map.json", "w") as f:
    json.dump({"id2dist": id2dist, "dist2id": dist2id}, f, indent=2)
print("Multiclass model saved to ./mbert_best_distortion_model")

In [ ]:
# ============================================================
# BLOCK 18 — MULTICLASS EVALUATION
# ============================================================
dist_model_eval = AutoModelForSequenceClassification.from_pretrained(
    "./best_distortion_model"
)
dist_model_eval.eval()

dist_pred = Trainer(
    model=dist_model_eval,
    args=TrainingArguments(
        output_dir="./eval_tmp_dist",
        per_device_eval_batch_size=8,
        report_to="none",
    ),
    data_collator=data_collator,
).predict(val_dist_dataset)

dist_y_true = dist_pred.label_ids
dist_y_pred = dist_pred.predictions.argmax(-1)

target_names_dist = [id2dist[i] for i in range(num_dist_labels)]

precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average="weighted", zero_division=0
)

precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average="macro", zero_division=0
)

precision_c, recall_c, f1_c, support_c = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average=None, zero_division=0
)

acc_d = accuracy_score(dist_y_true, dist_y_pred)

print("\n" + "="*55)
print("MULTICLASS EVALUATION RESULTS")
print("="*55)
print(f"Accuracy          : {acc_d:.4f}")
print(f"\nWeighted Average:")
print(f"  Precision : {precision_w:.4f}")
print(f"  Recall    : {recall_w:.4f}")
print(f"  F1        : {f1_w:.4f}")
print(f"\nMacro Average:")
print(f"  Precision : {precision_m:.4f}")
print(f"  Recall    : {recall_m:.4f}")
print(f"  F1        : {f1_m:.4f}")

print(f"\nPer-Class Results:")
print(f"{'Class':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 68)
for i, name in enumerate(target_names_dist):
    print(f"{name:<25} {precision_c[i]:>10.4f} {recall_c[i]:>10.4f} "
          f"{f1_c[i]:>10.4f} {int(support_c[i]):>10}")
print("-" * 68)

print("\n[MULTICLASS] Prediction distribution:")
unique, counts = np.unique(dist_y_pred, return_counts=True)
print({id2dist[int(u)]: int(c) for u, c in zip(unique, counts)})

print("\n[MULTICLASS] Confusion matrix:")
print(confusion_matrix(dist_y_true, dist_y_pred))

print("\n[MULTICLASS] Full classification report:")
print(classification_report(
    dist_y_true, dist_y_pred,
    target_names=target_names_dist,
    digits=4,
))

In [ ]:
# ============================================================
# BLOCK 19 — VISUALIZATION SETUP
# ============================================================
os.makedirs("./visualizations", exist_ok=True)
print("Visualization setup done.")


# ============================================================
# BLOCK 20 — RELOAD PREDICTIONS FOR VISUALIZATION
# ============================================================
with open("./best_binary_model/threshold.json") as f:
    best_threshold = json.load(f)["threshold"]
print(f"[VIZ] Binary threshold: {best_threshold}")

with open("./best_distortion_model/label_map.json") as f:
    label_map = json.load(f)
    id2dist_viz = {int(k): v for k, v in label_map["id2dist"].items()}
target_names_viz = [id2dist_viz[i] for i in range(len(id2dist_viz))]

bin_model_viz = AutoModelForSequenceClassification.from_pretrained(
    "./best_binary_model"
)
bin_viz_trainer = Trainer(
    model=bin_model_viz,
    args=TrainingArguments(
        output_dir="./eval_tmp_viz_bin",
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    eval_dataset=val_bin_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
bin_pred_viz   = bin_viz_trainer.predict(val_bin_dataset)
bin_y_true_viz = bin_pred_viz.label_ids
bin_logits_viz = bin_pred_viz.predictions
bin_y_pred_viz = bin_logits_viz.argmax(-1)
bin_probs_viz  = torch.softmax(
    torch.tensor(bin_logits_viz, dtype=torch.float), dim=-1
).numpy()
class1_probs_viz = bin_probs_viz[:, 1]
bin_preds_thr_viz = (class1_probs_viz >= best_threshold).astype(int)

dist_model_viz = AutoModelForSequenceClassification.from_pretrained(
    "./best_distortion_model"
)
dist_viz_trainer = Trainer(
    model=dist_model_viz,
    args=TrainingArguments(
        output_dir="./eval_tmp_viz_dist",
        per_device_eval_batch_size=8,
        report_to="none",
    ),
    eval_dataset=val_dist_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
dist_pred_viz  = dist_viz_trainer.predict(val_dist_dataset)
dist_y_true_viz = dist_pred_viz.label_ids
dist_logits_viz = dist_pred_viz.predictions
dist_y_pred_viz = dist_logits_viz.argmax(-1)
dist_probs_viz  = torch.softmax(
    torch.tensor(dist_logits_viz, dtype=torch.float), dim=-1
).numpy()

print("All predictions loaded for visualization.")

In [ ]:
# ============================================================
# BLOCK 21 — BINARY LOSS CURVES
# ============================================================
bin_train_losses, bin_val_losses, bin_epochs_log = [], [], []
for log in bin_trainer.state.log_history:
    if "loss" in log and "epoch" in log and "eval_loss" not in log:
        bin_train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        bin_val_losses.append(log["eval_loss"])
        bin_epochs_log.append(log["epoch"])

min_len = min(len(bin_train_losses), len(bin_val_losses))
bin_epochs_log   = bin_epochs_log[:min_len]
bin_train_losses = bin_train_losses[:min_len]
bin_val_losses   = bin_val_losses[:min_len]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(bin_epochs_log, bin_train_losses, marker="o",
        label="Training Loss", color="#4C72B0", linewidth=2)
ax.plot(bin_epochs_log, bin_val_losses, marker="s",
        label="Validation Loss", color="#DD8452", linewidth=2)
best_idx = int(np.argmin(bin_val_losses))
ax.axvline(x=bin_epochs_log[best_idx], color="gray", linestyle="--",
           linewidth=1.2, label=f"Best epoch ({int(bin_epochs_log[best_idx])})")
ax.set_title("Binary Model — Training vs Validation Loss",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss",  fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./visualizations/binary_loss_curves.png", dpi=150)
plt.show()
print("Saved: binary_loss_curves.png")


# ============================================================
# BLOCK 22 — BINARY CONFUSION MATRIX
# ============================================================
cm_bin = confusion_matrix(bin_y_true_viz, bin_preds_thr_viz)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_bin,
    display_labels=["No Distortion", "Distorted"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
for i in range(cm_bin.shape[0]):
    for j in range(cm_bin.shape[1]):
        pct = cm_bin[i, j] / cm_bin[i].sum() * 100
        ax.text(j, i + 0.3, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=10, color="black")
ax.set_title(f"Binary Model — Confusion Matrix\n(threshold={best_threshold})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("./visualizations/binary_confusion_matrix.png", dpi=150)
plt.show()
print("Saved: binary_confusion_matrix.png")


# ============================================================
# BLOCK 23 — BINARY ROC-AUC
# ============================================================
fpr_viz, tpr_viz, thr_roc = roc_curve(bin_y_true_viz, class1_probs_viz)
roc_auc_viz   = auc(fpr_viz, tpr_viz)
best_thr_idx  = np.argmin(np.abs(thr_roc - best_threshold))

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_viz, tpr_viz, color="#4C72B0", linewidth=2.5,
        label=f"ROC Curve (AUC = {roc_auc_viz:.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random Classifier")
ax.scatter(fpr_viz[best_thr_idx], tpr_viz[best_thr_idx],
           color="#DD8452", s=120, zorder=5,
           label=f"Threshold={best_threshold}")
ax.set_title("Binary Model — ROC-AUC Curve", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.legend(fontsize=10, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./visualizations/binary_roc_auc.png", dpi=150)
plt.show()
print("Saved: binary_roc_auc.png")


In [ ]:
# ============================================================
# BLOCK 24 — MULTICLASS LOSS CURVES
# ============================================================
dist_train_losses, dist_val_losses, dist_epochs_log = [], [], []
for log in dist_trainer.state.log_history:
    if "loss" in log and "epoch" in log and "eval_loss" not in log:
        dist_train_losses.append(log["loss"])
    if "eval_loss" in log and "epoch" in log:
        dist_val_losses.append(log["eval_loss"])
        dist_epochs_log.append(log["epoch"])

min_len = min(len(dist_train_losses), len(dist_val_losses))
dist_epochs_log   = dist_epochs_log[:min_len]
dist_train_losses = dist_train_losses[:min_len]
dist_val_losses   = dist_val_losses[:min_len]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(dist_epochs_log, dist_train_losses, marker="o",
        label="Training Loss", color="#4C72B0", linewidth=2)
ax.plot(dist_epochs_log, dist_val_losses, marker="s",
        label="Validation Loss", color="#DD8452", linewidth=2)
best_dist_idx = int(np.argmin(dist_val_losses))
ax.axvline(x=dist_epochs_log[best_dist_idx], color="gray", linestyle="--",
           linewidth=1.2,
           label=f"Best epoch ({int(dist_epochs_log[best_dist_idx])})")
ax.set_title("Multiclass Model — Training vs Validation Loss",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss",  fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./visualizations/multiclass_loss_curves.png", dpi=150)
plt.show()
print("Saved: multiclass_loss_curves.png")


# ============================================================
# BLOCK 25 — MULTICLASS CONFUSION MATRIX
# ============================================================
cm_dist = confusion_matrix(dist_y_true_viz, dist_y_pred_viz)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dist,
    display_labels=target_names_viz
)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=45)
ax.set_title("Multiclass Model — Confusion Matrix",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("./visualizations/multiclass_confusion_matrix.png", dpi=150)
plt.show()
print("Saved: multiclass_confusion_matrix.png")


# ============================================================
# BLOCK 26 — MULTICLASS ROC-AUC
# ============================================================
num_viz_labels = len(target_names_viz)
y_true_bin_viz = label_binarize(
    dist_y_true_viz, classes=list(range(num_viz_labels))
)
colors = plt.cm.tab10(np.linspace(0, 1, num_viz_labels))

fig, ax = plt.subplots(figsize=(10, 7))
for i, (label, color) in enumerate(zip(target_names_viz, colors)):
    fpr_i, tpr_i, _ = roc_curve(y_true_bin_viz[:, i], dist_probs_viz[:, i])
    auc_i = auc(fpr_i, tpr_i)
    ax.plot(fpr_i, tpr_i, linewidth=2, color=color,
            label=f"{label} (AUC={auc_i:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random")
ax.set_title("Multiclass Model — ROC-AUC (One vs Rest)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./visualizations/multiclass_roc_auc.png", dpi=150)
plt.show()
print("Saved: multiclass_roc_auc.png")


In [ ]:
# ============================================================
# BLOCK 27 — FINAL TRAINING SUMMARY
# ============================================================
print("\n" + "="*55)
print("FINAL TRAINING SUMMARY")
print("="*55)
print(f"\n[BINARY] Best threshold : {best_threshold}")
print(f"[BINARY] ROC-AUC        : {roc_auc_viz:.4f}")
print("\n[BINARY] Classification Report (at best threshold):")
print(classification_report(bin_y_true_viz, bin_preds_thr_viz,
      target_names=["No Distortion", "Distorted"], digits=4))
print("\n[MULTICLASS] Classification Report:")
print(classification_report(dist_y_true_viz, dist_y_pred_viz,
      target_names=target_names_viz, digits=4))
print("\nVisualizations saved to ./visualizations/")
print("="*55)